In [1]:
from pathlib import Path
import gc
import importlib
import itertools
import json
import os
import re
import sys
import tarfile
import time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
HF_COMPAT_DIR = PROJECT_ROOT / '.hf_compat'
if HF_COMPAT_DIR.exists():
    compat_path = str(HF_COMPAT_DIR)
    if compat_path in sys.path:
        sys.path.remove(compat_path)
    sys.path.insert(0, compat_path)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import sacrebleu
import torch
from huggingface_hub import snapshot_download
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOCAL_418M = Path(r'C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636')
MODEL_1_2B_DIR = PROJECT_ROOT / 'models' / 'huggingface' / 'm2m100_1.2B'
FLORES_ARCHIVE = PROJECT_ROOT / 'data' / 'external' / 'trusted_benchmarks' / 'flores200_dataset.tar.gz'
RESULT_DIR = PROJECT_ROOT / 'results' / 'fourlang_base_model_selection' / 'beam5_accuracy'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_1_2B_DIR.mkdir(parents=True, exist_ok=True)
LANG_MEMBERS = {'en': 'eng_Latn', 'uz': 'uzn_Latn', 'ru': 'rus_Cyrl', 'zh': 'zho_Hans'}
LANG_NAMES = {'en': 'English', 'uz': 'Uzbek', 'ru': 'Russian', 'zh': 'Chinese'}
NUM_PAIRS_PER_DIRECTION = 50
SEED = 42
MAX_SOURCE_LENGTH = 192
MAX_NEW_TOKENS = 160
NUM_BEAMS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == 'cuda' else torch.float32)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert DEVICE == 'cuda', 'CUDA GPU was not detected.'
assert (LOCAL_418M / 'config.json').exists(), f'Local 418M model not found: {LOCAL_418M}'
assert FLORES_ARCHIVE.exists(), f'FLORES archive not found: {FLORES_ARCHIVE}'
print('Device:', DEVICE, 'dtype:', DTYPE)
print('418M:', LOCAL_418M)
print('1.2B download directory:', MODEL_1_2B_DIR)
print('Results:', RESULT_DIR)


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda dtype: torch.bfloat16
418M: C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636
1.2B download directory: D:\dev\projects\fourlang_translation\models\huggingface\m2m100_1.2B
Results: D:\dev\projects\fourlang_translation\results\fourlang_base_model_selection\beam5_accuracy


In [2]:
required_1_2b_files = [
    'config.json',
    'generation_config.json',
    'pytorch_model.bin',
    'sentencepiece.bpe.model',
    'special_tokens_map.json',
    'tokenizer_config.json',
    'vocab.json',
]
complete_1_2b = all((MODEL_1_2B_DIR / name).exists() for name in required_1_2b_files)
if not complete_1_2b:
    print('Downloading only the required M2M100 1.2B files to D drive. Expected download: about 5 GB.')
    snapshot_download(
        repo_id='facebook/m2m100_1.2B',
        local_dir=str(MODEL_1_2B_DIR),
        allow_patterns=required_1_2b_files,
        max_workers=2,
    )
else:
    print('M2M100 1.2B is already complete; download skipped.')
missing = [name for name in required_1_2b_files if not (MODEL_1_2B_DIR / name).exists()]
assert not missing, f'Incomplete 1.2B download, missing: {missing}'
assert (MODEL_1_2B_DIR / 'pytorch_model.bin').stat().st_size > 4_000_000_000, 'The 1.2B weight file is incomplete.'
print('1.2B model ready:', MODEL_1_2B_DIR)


M2M100 1.2B is already complete; download skipped.
1.2B model ready: D:\dev\projects\fourlang_translation\models\huggingface\m2m100_1.2B


In [3]:
def read_tar_lines(archive_path, member_name):
    normalized = member_name.lstrip('./')
    with tarfile.open(archive_path, 'r:gz') as archive:
        matches = [m for m in archive.getmembers() if m.name.lstrip('./') == normalized]
        if not matches:
            matches = [m for m in archive.getmembers() if m.name.lstrip('./').endswith(normalized)]
        assert len(matches) == 1, f'Expected one archive member for {normalized}, found {len(matches)}'
        handle = archive.extractfile(matches[0])
        assert handle is not None
        return handle.read().decode('utf-8').splitlines()

language_lines = {}
for code, member in LANG_MEMBERS.items():
    language_lines[code] = read_tar_lines(FLORES_ARCHIVE, f'flores200_dataset/devtest/{member}.devtest')
line_counts = {code: len(lines) for code, lines in language_lines.items()}
assert len(set(line_counts.values())) == 1, line_counts
rng = np.random.default_rng(SEED)
sample_indices = sorted(rng.choice(next(iter(line_counts.values())), size=NUM_PAIRS_PER_DIRECTION, replace=False).tolist())
rows = []
for src_lang, tgt_lang in itertools.permutations(LANG_MEMBERS, 2):
    for pair_id in sample_indices:
        rows.append({
            'eval_id': f'{src_lang}-{tgt_lang}-{pair_id:04d}',
            'benchmark': 'flores_devtest',
            'pair_id': pair_id,
            'src_lang': src_lang,
            'tgt_lang': tgt_lang,
            'source': language_lines[src_lang][pair_id],
            'reference': language_lines[tgt_lang][pair_id],
        })
benchmark_df = pd.DataFrame(rows).sort_values('eval_id').reset_index(drop=True)
assert len(benchmark_df) == 12 * NUM_PAIRS_PER_DIRECTION
assert not benchmark_df['eval_id'].duplicated().any()
benchmark_df.to_csv(RESULT_DIR / 'flores_fourlang_600.csv', index=False, encoding='utf-8-sig')
display(benchmark_df.groupby(['src_lang', 'tgt_lang']).size().rename('samples').reset_index())
display(benchmark_df.head())


,src_lang,tgt_lang,samples
0,en,ru,50
1,en,uz,50
2,en,zh,50
3,ru,en,50
4,ru,uz,50
5,ru,zh,50
6,uz,en,50
7,uz,ru,50
8,uz,zh,50
9,zh,en,50


,eval_id,benchmark,pair_id,src_lang,tgt_lang,source,reference
0,en-ru-0063,flores_devtest,63,en,ru,Historians have criticized past FBI policies f...,"Историки критикуют прошлую политику ФБР за то,..."
1,en-ru-0068,flores_devtest,68,en,ru,U.S. President George W. Bush arrived in Singa...,Президент США Джордж Буш прибыл в Сингапур утр...
2,en-ru-0083,flores_devtest,83,en,ru,"The Ninth Ward, which saw flooding as high as ...","Девятый административный район, затопленный на..."
3,en-ru-0085,flores_devtest,85,en,ru,Commons Administrator Adam Cuerden expressed h...,Администратор Commons Адам Керден выразил своё...
4,en-ru-0091,flores_devtest,91,en,ru,The scientists were able to conclude that the ...,"Ученые смогли сделать вывод, что темная матери..."


In [4]:
MODEL_SPECS = [
    {'model_name': 'm2m100_418m', 'path': LOCAL_418M, 'license': 'MIT', 'hub_id': 'facebook/m2m100_418M'},
    {'model_name': 'm2m100_1_2b', 'path': MODEL_1_2B_DIR, 'license': 'MIT', 'hub_id': 'facebook/m2m100_1.2B'},
]

def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

def word_tokens(text):
    return re.findall(r"[^\W_]+(?:['’ʻʼ-][^\W_]+)*", str(text).casefold(), flags=re.UNICODE)

def repeated_phrase(text):
    tokens = word_tokens(text)
    for width in range(1, min(6, len(tokens) // 3 + 1)):
        for start in range(len(tokens) - 3 * width + 1):
            phrase = tokens[start:start + width]
            if phrase == tokens[start + width:start + 2 * width] == tokens[start + 2 * width:start + 3 * width]:
                return True
    return False

def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        str(model_path),
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    model.to(DEVICE)
    model.eval()
    model.config.use_cache = True
    return model, tokenizer

@torch.inference_mode()
def translate_one(model, tokenizer, text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    started = time.perf_counter()
    inputs = tokenizer(str(text), return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LENGTH).to(DEVICE)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        early_stopping=True,
        length_penalty=1.0,
        repetition_penalty=1.10,
        no_repeat_ngram_size=3,
    )
    synchronize()
    seconds = time.perf_counter() - started
    prediction = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return prediction, seconds, int(generated.shape[-1])

print('Model and translation helpers are ready.')


Model and translation helpers are ready.


In [5]:
runtime_records = []
for spec in MODEL_SPECS:
    model_name = spec['model_name']
    output_file = RESULT_DIR / f'predictions_{model_name}.csv'
    existing = pd.read_csv(output_file, keep_default_na=False) if output_file.exists() else pd.DataFrame()
    completed = set(existing['eval_id'].astype(str)) if len(existing) else set()
    pending = benchmark_df[~benchmark_df['eval_id'].astype(str).isin(completed)]
    print(model_name, 'completed:', len(completed), 'pending:', len(pending), flush=True)
    if not len(pending):
        continue
    if 'model' in globals():
        del model
    if 'tokenizer' in globals():
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'GPU free before loading {model_name}: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GB', flush=True)
    print('Loading model:', spec['path'], flush=True)
    load_started = time.perf_counter()
    model, tokenizer = load_model_and_tokenizer(spec['path'])
    load_seconds = time.perf_counter() - load_started
    print(f'Model loaded in {load_seconds:.1f} seconds. Starting warm-up...', flush=True)
    _ = translate_one(model, tokenizer, 'Hello.', 'en', 'uz')
    print('Warm-up complete. Translation progress starts now.', flush=True)
    new_rows = []
    for row in tqdm(pending.itertuples(index=False), total=len(pending), desc=model_name, mininterval=0.5, dynamic_ncols=True):
        prediction, total_seconds, generated_tokens = translate_one(model, tokenizer, row.source, row.src_lang, row.tgt_lang)
        new_rows.append({
            **row._asdict(),
            'model_name': model_name,
            'prediction': prediction,
            'total_seconds': total_seconds,
            'generated_tokens': generated_tokens,
            'has_repetition': repeated_phrase(prediction),
            'hit_max_tokens': generated_tokens >= MAX_NEW_TOKENS,
        })
        if len(new_rows) % 10 == 0:
            saved = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
            saved.drop_duplicates('eval_id', keep='last').sort_values('eval_id').to_csv(output_file, index=False, encoding='utf-8-sig')
    saved = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    saved = saved.drop_duplicates('eval_id', keep='last').sort_values('eval_id')
    saved.to_csv(output_file, index=False, encoding='utf-8-sig')
    runtime = {
        'model_name': model_name,
        'hub_id': spec['hub_id'],
        'license': spec['license'],
        'parameters': int(sum(p.numel() for p in model.parameters())),
        'load_seconds': load_seconds,
        'peak_vram_gb': torch.cuda.max_memory_allocated() / 1024**3,
    }
    runtime_records.append(runtime)
    (RESULT_DIR / f'runtime_{model_name}.json').write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding='utf-8')
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
print('Evaluation cell finished. Re-run it after any interruption; completed rows will be skipped.')


m2m100_418m completed: 0 pending: 600
GPU free before loading m2m100_418m: 6.88/7.96 GB
Loading model: C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636
Model loaded in 4.3 seconds. Starting warm-up...
Warm-up complete. Translation progress starts now.


m2m100_418m: 100%|██████████| 600/600 [03:14<00:00,  3.09it/s]


m2m100_1_2b completed: 0 pending: 600
GPU free before loading m2m100_1_2b: 6.76/7.96 GB
Loading model: D:\dev\projects\fourlang_translation\models\huggingface\m2m100_1.2B
Model loaded in 6.3 seconds. Starting warm-up...
Warm-up complete. Translation progress starts now.


m2m100_1_2b: 100%|██████████| 600/600 [05:46<00:00,  1.73it/s]

Evaluation cell finished. Re-run it after any interruption; completed rows will be skipped.


In [6]:
def truthy(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def compute_metrics(frame):
    records = []
    for (model_name, src_lang, tgt_lang), group in frame.groupby(['model_name', 'src_lang', 'tgt_lang']):
        predictions = group['prediction'].astype(str).tolist()
        references = group['reference'].astype(str).tolist()
        bleu_tokenizer = 'zh' if tgt_lang == 'zh' else '13a'
        records.append({
            'model_name': model_name,
            'direction': f'{src_lang}-{tgt_lang}',
            'src_lang': src_lang,
            'tgt_lang': tgt_lang,
            'samples': len(group),
            'bleu': sacrebleu.corpus_bleu(predictions, [references], tokenize=bleu_tokenizer).score,
            'chrf2': sacrebleu.corpus_chrf(predictions, [references], word_order=2).score,
            'repetition_percent': float(truthy(group['has_repetition']).mean() * 100),
            'hit_max_tokens_percent': float(truthy(group['hit_max_tokens']).mean() * 100),
            'latency_mean_seconds': float(pd.to_numeric(group['total_seconds']).mean()),
            'latency_p95_seconds': float(pd.to_numeric(group['total_seconds']).quantile(.95)),
        })
    return pd.DataFrame(records)

prediction_frames = []
for spec in MODEL_SPECS:
    path = RESULT_DIR / f"predictions_{spec['model_name']}.csv"
    assert path.exists(), f'Missing predictions: {path}'
    frame = pd.read_csv(path, keep_default_na=False)
    assert len(frame) == len(benchmark_df), f"{spec['model_name']} incomplete: {len(frame)}/{len(benchmark_df)}"
    prediction_frames.append(frame)
all_predictions = pd.concat(prediction_frames, ignore_index=True)
metrics = compute_metrics(all_predictions)
metrics.to_csv(RESULT_DIR / 'metrics_418m_vs_1_2b_fourlang.csv', index=False, encoding='utf-8-sig')
display(metrics.round(4))


,model_name,direction,src_lang,tgt_lang,samples,bleu,chrf2,repetition_percent,hit_max_tokens_percent,latency_mean_seconds,latency_p95_seconds
0,m2m100_1_2b,en-ru,en,ru,50,29.1910,54.0162,0.0,0.0,0.5604,0.8705
1,m2m100_1_2b,en-uz,en,uz,50,0.3830,13.9508,0.0,0.0,0.5167,0.7521
2,m2m100_1_2b,en-zh,en,zh,50,34.5805,23.1136,0.0,0.0,0.4735,0.7469
3,m2m100_1_2b,ru-en,ru,en,50,35.0850,59.6190,0.0,0.0,0.5171,0.7409
4,m2m100_1_2b,ru-uz,ru,uz,50,0.3447,11.6773,2.0,0.0,0.5811,0.8556
5,m2m100_1_2b,ru-zh,ru,zh,50,29.6640,19.5259,0.0,0.0,0.4790,0.7185
6,m2m100_1_2b,uz-en,uz,en,50,2.2183,20.7388,0.0,0.0,0.7475,1.7707
7,m2m100_1_2b,uz-ru,uz,ru,50,1.2399,18.0590,0.0,0.0,0.7532,1.3837
8,m2m100_1_2b,uz-zh,uz,zh,50,4.9953,4.7236,0.0,0.0,0.6395,1.1241
9,m2m100_1_2b,zh-en,zh,en,50,24.1026,52.9165,0.0,0.0,0.5032,0.7927


In [7]:
base = metrics[metrics.model_name == 'm2m100_418m'].set_index('direction')
candidate = metrics[metrics.model_name == 'm2m100_1_2b'].set_index('direction')
comparison = candidate[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']].subtract(
    base[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']]
).add_prefix('delta_1_2b_minus_418m_').reset_index()
comparison.to_csv(RESULT_DIR / 'comparison_1_2b_minus_418m.csv', index=False, encoding='utf-8-sig')
display(comparison.round(4))

summary = metrics.groupby('model_name').agg(
    mean_bleu=('bleu', 'mean'),
    mean_chrf2=('chrf2', 'mean'),
    median_chrf2=('chrf2', 'median'),
    mean_repetition_percent=('repetition_percent', 'mean'),
    mean_hit_max_tokens_percent=('hit_max_tokens_percent', 'mean'),
    mean_latency_p95_seconds=('latency_p95_seconds', 'mean'),
).reset_index()
en_uz = metrics[metrics.direction.isin(['en-uz', 'uz-en'])].groupby('model_name')[['bleu', 'chrf2']].mean().add_prefix('en_uz_mean_').reset_index()
summary = summary.merge(en_uz, on='model_name', how='left')
summary.to_csv(RESULT_DIR / 'model_selection_summary.csv', index=False, encoding='utf-8-sig')
display(summary.round(4))

wins = comparison['delta_1_2b_minus_418m_chrf2'] > 0
print('1.2B chrF++ wins:', int(wins.sum()), '/', len(wins))
print('1.2B mean chrF++ delta:', round(comparison['delta_1_2b_minus_418m_chrf2'].mean(), 4))
print('This benchmark is a selection screen, not production acceptance.')


,direction,delta_1_2b_minus_418m_bleu,delta_1_2b_minus_418m_chrf2,delta_1_2b_minus_418m_repetition_percent,delta_1_2b_minus_418m_hit_max_tokens_percent,delta_1_2b_minus_418m_latency_p95_seconds
0,en-ru,4.9221,4.0577,0.0,0.0,0.3779
1,en-uz,-0.1334,-1.5716,-2.0,0.0,0.2648
2,en-zh,2.9924,2.5210,0.0,0.0,0.3661
3,ru-en,7.6917,5.2356,0.0,0.0,0.3356
4,ru-uz,0.0010,-3.9345,0.0,0.0,0.3409
5,ru-zh,1.8847,0.3073,0.0,0.0,0.2596
6,uz-en,0.5943,-1.2942,0.0,0.0,0.9469
7,uz-ru,0.1041,-0.7944,0.0,0.0,0.6590
8,uz-zh,0.3919,-0.6593,0.0,0.0,0.6055
9,zh-en,3.7832,3.5820,0.0,0.0,0.2994


,model_name,mean_bleu,mean_chrf2,median_chrf2,mean_repetition_percent,mean_hit_max_tokens_percent,mean_latency_p95_seconds,en_uz_mean_bleu,en_uz_mean_chrf2
0,m2m100_1_2b,14.8752,27.7509,20.1323,0.1667,0.0,0.9621,1.3006,17.3448
1,m2m100_418m,12.7557,27.1442,19.9056,0.8333,0.0,0.5298,1.0702,18.7776


1.2B chrF++ wins: 6 / 12
1.2B mean chrF++ delta: 0.6067
This benchmark is a selection screen, not production acceptance.


In [8]:
diagnostics = all_predictions.copy()
diagnostics['length_ratio'] = diagnostics['prediction'].astype(str).str.len() / diagnostics['reference'].astype(str).str.len().clip(lower=1)
problem_rows = diagnostics[truthy(diagnostics['has_repetition']) | truthy(diagnostics['hit_max_tokens']) | (diagnostics.length_ratio < 0.35) | (diagnostics.length_ratio > 2.5)]
problem_rows.to_csv(RESULT_DIR / 'diagnostic_problem_rows.csv', index=False, encoding='utf-8-sig')
print('Problem rows:', len(problem_rows), '/', len(diagnostics))
display(problem_rows[['model_name', 'src_lang', 'tgt_lang', 'source', 'reference', 'prediction', 'length_ratio', 'has_repetition', 'hit_max_tokens']].head(30))

samples = diagnostics.groupby(['model_name', 'src_lang', 'tgt_lang'], group_keys=False).sample(n=2, random_state=42)
samples.to_csv(RESULT_DIR / 'qualitative_samples.csv', index=False, encoding='utf-8-sig')
display(samples[['model_name', 'src_lang', 'tgt_lang', 'source', 'reference', 'prediction']].sort_values(['model_name', 'src_lang', 'tgt_lang']))

manifest = {
    'benchmark': 'FLORES-200 devtest',
    'benchmark_usage': 'evaluation_only',
    'benchmark_license': 'CC-BY-SA-4.0',
    'languages': LANG_MEMBERS,
    'pairs_per_direction': NUM_PAIRS_PER_DIRECTION,
    'total_directions': 12,
    'models': [{k: str(v) if isinstance(v, Path) else v for k, v in spec.items()} for spec in MODEL_SPECS],
    'decode': {'num_beams': NUM_BEAMS, 'early_stopping': True, 'length_penalty': 1.0, 'repetition_penalty': 1.10, 'no_repeat_ngram_size': 3, 'max_source_length': MAX_SOURCE_LENGTH, 'max_new_tokens': MAX_NEW_TOKENS},
}
(RESULT_DIR / 'evaluation_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('All artifacts saved to:', RESULT_DIR)


Problem rows: 21 / 1200


,model_name,src_lang,tgt_lang,source,reference,prediction,length_ratio,has_repetition,hit_max_tokens
89,m2m100_418m,en,uz,If you booked your flights and accommodation f...,Agar siz kechiktirish e'lon qilinishidan avval...,Siz 2020 o‘shaytlar bilan bilan bilan bo‘qayti...,0.477124,True,False
239,m2m100_418m,ru,uz,Если вы забронировали авиабилеты и проживание ...,Agar siz kechiktirish e'lon qilinishidan avval...,Siz 2020 yilda uçuşlar və qaytarlar bilan bila...,0.699346,True,False
434,m2m100_418m,uz,zh,Chako mintaqasida Guaykuru va Payagua kabi mah...,查科 (Chaco) 地区是瓜伊库鲁 (Guaycuru) 和帕亚瓜 (Payagua) 等...,"古人云:《古兰经》说:‘古人有罪,有罪。",0.285714,False,False
579,m2m100_418m,zh,uz,以虚拟方式分享实地考察，也是回顾行程以及与以后班级分享经验的好办法。,Ekskursiyalarni virtual tarzda bo'lishish sayo...,Masiya viziyatini virtualni bir şekilde paylaş...,1.000000,True,False
582,m2m100_418m,zh,uz,又因为相对偏僻，“廷巴克图”被用来比喻遥远的异国他乡。,Yetishish imkoniyati nisbatan kamligi bilan bi...,«Tinbarktur» o‘yadi bilan bilan bilan qaytadi.,0.356589,True,False
589,m2m100_418m,zh,uz,如果你在宣布推迟之前预订了 2020 年的航班和住宿，则可能会面临棘手的情况。,Agar siz kechiktirish e'lon qilinishidan avval...,2020 yildan uçuşlar ve konvaziyiz o‘ziz bilan ...,0.483660,True,False
672,m2m100_1_2b,en,uz,Children are placed in Foster Care for a wide ...,"Bolalar e'tiborsizlikdan tortib, zo'ravonlik v...","Foster Care o‘z o‘ladi, o’ldi, oʻldi.",0.276119,False,False
805,m2m100_1_2b,ru,uz,"Человек, работающий в гараже недалеко от того ...",Ko'ngilsiz hodisa yuz bergan joy yaqinidagi ga...,"O‘z, o‘z o‘zi, o’z oʻz o`z o ́z o'z o.o.o., o‘...",0.377660,True,False
811,m2m100_1_2b,ru,uz,Для запуска в космос спутника или телескопа не...,Sun'iy yo'ldosh yoki teleskopni kosmosga joyla...,"Satellite, teleskop o‘z orbitni qilmadi.",0.333333,False,False
812,m2m100_1_2b,ru,uz,Самки обычно находятся друг с другом в родстве...,"Urg'ochilar, odatda, katta oilaning opa-singil...","Men o‘z o‘ladi, o’ldi, o‘lo‘ldi.",0.310680,False,False


,model_name,src_lang,tgt_lang,source,reference,prediction
613,m2m100_1_2b,en,ru,There were no large forests in the land of Can...,"На земле Ханаана не было больших лесов, поэтом...","В Ханаанской земле не было больших лесов, поэт..."
639,m2m100_1_2b,en,ru,If you booked your flights and accommodation f...,Если вы забронировали авиабилеты и проживание ...,Если вы забронировали рейсы и проживание на 20...
679,m2m100_1_2b,en,uz,Sharing a field trip virtually is also a great...,Ekskursiyalarni virtual tarzda bo'lishish sayo...,"Bu yolda o‘z bilan o‘ladi, o‘zi o‘zuldi, o’z o..."
688,m2m100_1_2b,en,uz,Saltwater Crocodiles do not actively live in t...,"Sho'r suv timsohlari okeanda ko'p uchramaydi, ...",Saltwater Crocodiles aktiv o‘zayda yaşaymazlar...
701,m2m100_1_2b,en,zh,U.S. President George W. Bush arrived in Singa...,美国总统乔治·沃克·布什于 11 月 16 日上午抵达新加坡，开始为期一周的亚洲之行。,"美国总统布什11月16日上午抵达新加坡,开始为期一周的亚洲之旅。"
749,m2m100_1_2b,en,zh,Inland waterways can be a good theme to base a...,内陆水道可以作为假期游玩的一个不错的主题。,内陆水道可以是一个很好的主题来建立一个假期周围。
799,m2m100_1_2b,ru,en,Внутренние водные пути могут стать хорошей тем...,Inland waterways can be a good theme to base a...,Inland waterways can be a good theme for any c...
766,m2m100_1_2b,ru,en,Её всепроникающая сила коснулась каждого - от ...,Its all-pervading power affected everyone from...,"Her all-pervading power touched everyone, from..."
835,m2m100_1_2b,ru,uz,Зима может быть обманчиво холодной: температур...,Qish chalg'itadigan sovuq bo'lishi mumkin: har...,Bilmiz o‘z o‘zi qilmizdir: temperatur o‘zini q...
806,m2m100_1_2b,ru,uz,Начиная с 1988 года урны для избирательных бюл...,1988-yildan beri ovoz beruvchilar va kuzatuvch...,"1988-cından o‘z urnalarni qilmadi, o‘zimlarni ..."


All artifacts saved to: D:\dev\projects\fourlang_translation\results\fourlang_base_model_selection\beam5_accuracy
